# Isolate RNA-seq: plasmid, pga and MS2 load

Figures 4C, 4D, S12A, S12B, S13

Environment: `bulk_R_env` (`envs/bulk_R_env.yml`)

Inputs:
- `isolates_RNAseq_htseq_raw_counts.csv` (GSE349085), in `data_dir/GSE349085/`
- `data/isolates_RNAseq_feature_annotation.csv` (feature → gene name and replicon)

In [ ]:
suppressPackageStartupMessages({
  library(yaml)
  library(DESeq2)
  library(ggplot2)
  library(dplyr)
  library(ggrepel)
  library(matrixStats)
  library(patchwork)
})

ROOT <- ".."
config <- read_yaml(file.path(ROOT, "config.yaml"))
DATA_DIR <- file.path(config$data_dir, "GSE349085")
OUT_DIR  <- config$out_dir
FIG_DIR  <- file.path(OUT_DIR, "figures")
SRC_DIR  <- file.path(OUT_DIR, "source_data")
for (d in c(FIG_DIR, SRC_DIR)) dir.create(d, recursive = TRUE, showWarnings = FALSE)

CONTROL_SAMPLES <- c("c1", "c2", "c3")
PLAQUE_POSITIVE <- c("i6", "i8")
PLASMID_NULL_CUT <- 1e-4
MIN_COUNT <- 10
MIN_SAMPLES <- 3
PAL <- c("Isolate" = "#FF7F0E", "Control" = "#1F77B4")

set_plot <- function(w, h) options(repr.plot.width = w, repr.plot.height = h)

save_fig <- function(p, name, w, h, dpi = 300) {
  set_plot(w, h)
  print(p)
  ggsave(file.path(FIG_DIR, paste0(name, ".png")), p, width = w, height = h, dpi = dpi, bg = "white")
  ggsave(file.path(FIG_DIR, paste0(name, ".pdf")), p, width = w, height = h, bg = "white", device = cairo_pdf)
}

In [ ]:
counts <- read.csv(file.path(DATA_DIR, "isolates_RNAseq_htseq_raw_counts.csv"), check.names = FALSE)
ann <- read.csv(file.path(ROOT, "data", "isolates_RNAseq_feature_annotation.csv"))

feature_id <- counts[[1]]
counts <- as.matrix(counts[, -1])
stopifnot(all(feature_id %in% ann$feature_id))
ann <- ann[match(feature_id, ann$feature_id), ]
rownames(counts) <- ann$gene

ms2_pos <- ann$gene[ann$replicon %in% "MS2_positive"]
ms2_neg <- ann$gene[ann$replicon %in% "MS2_negative"]
plasmid_features <- ann$gene[ann$replicon %in% "plasmid"]
chrom_features   <- ann$gene[ann$replicon %in% "chromosome"]
host_features    <- c(chrom_features, plasmid_features)
cat("chromosome:", length(chrom_features), "| plasmid:", length(plasmid_features),
    "| MS2:", length(c(ms2_pos, ms2_neg)), "\n")

csum <- function(f) unname(colSums(counts[f, , drop = FALSE]))
comp <- data.frame(sample = colnames(counts), chromosome = csum(chrom_features),
                   plasmid = csum(plasmid_features), MS2_pos = csum(ms2_pos), MS2_neg = csum(ms2_neg)) %>%
  mutate(host_total = chromosome + plasmid,
         plasmid_frac = plasmid / host_total,
         ms2     = log10(1e6 * MS2_pos / host_total + 1),
         ms2_neg = log10(1e6 * MS2_neg / host_total + 1))

cts_host <- counts[host_features, , drop = FALSE]
cts_host <- cts_host[rowSums(cts_host >= MIN_COUNT) >= MIN_SAMPLES, , drop = FALSE]
plasmid_kept <- intersect(plasmid_features, rownames(cts_host))
chrom_kept   <- intersect(chrom_features,   rownames(cts_host))

dds <- DESeqDataSetFromMatrix(cts_host, colData = data.frame(row.names = colnames(cts_host)), design = ~ 1)
dds <- estimateSizeFactors(dds, controlGenes = which(rownames(dds) %in% chrom_kept))
cat("genes kept:", nrow(dds), "\n")

raw_host <- counts(dds, normalized = FALSE)
host_cpm <- 1e6 * t(t(raw_host) / colSums(raw_host))
mean_log_cpm <- function(genes) colMeans(log10(host_cpm[genes, comp$sample, drop = FALSE] + 1))

In [ ]:
PILUS_REGEX <- "^tr[ab][A-Z]?[0-9]*(_[0-9]+)?(\\.[0-9]+)?$"
pilus_genes <- c(grep(PILUS_REGEX, plasmid_kept, value = TRUE, ignore.case = TRUE),
                 grep("^(finO|ylpA)(_[0-9]+)?$", plasmid_kept, value = TRUE))
other_genes <- setdiff(plasmid_kept, pilus_genes)
pga_genes   <- grep("^pga[ABCD](_[0-9]+)?$", rownames(dds), value = TRUE)
cat("expressed plasmid genes (Fig 4C):", length(plasmid_kept), "\n",
    "pilus-associated (Fig S12A, left):", length(pilus_genes), "\n",
    "non-pilus (Fig S12A, right):", length(other_genes), "\n",
    "pga genes (Fig 4D):", paste(pga_genes, collapse = ", "), "\n")

fig_df <- comp %>%
  mutate(plasmid_state = ifelse(plasmid_frac <= PLASMID_NULL_CUT, "plasmid_null", "plasmid_pos"),
         plasmid_log10cpm = mean_log_cpm(plasmid_kept),
         pga_log10cpm     = mean_log_cpm(pga_genes),
         pilus_log10cpm   = mean_log_cpm(pilus_genes),
         other_log10cpm   = mean_log_cpm(other_genes),
         group = factor(ifelse(sample %in% CONTROL_SAMPLES, "Control", "Isolate"),
                        levels = c("Isolate", "Control")),
         fade  = ifelse(sample %in% PLAQUE_POSITIVE, "faded", "normal"),
         fitted = !sample %in% c(CONTROL_SAMPLES, PLAQUE_POSITIVE))
fit_set <- fig_df$fitted
cat("fit set n =", sum(fit_set), "of", nrow(fig_df), "\n")

fit_stats <- function(y, x = fig_df$ms2, keep = fit_set) {
  f <- summary(lm(y[keep] ~ x[keep]))
  c(n = sum(keep), R2 = f$r.squared, p = coef(f)[2, 4])
}

base_theme <- theme_bw(base_size = 15) +
  theme(panel.grid.minor = element_blank(),
        panel.border = element_rect(colour = "black", fill = NA, linewidth = .7),
        legend.position = "bottom", legend.title = element_blank(),
        legend.text = element_text(size = 16),
        legend.key.size = unit(1.1, "lines"),
        axis.text = element_text(colour = "black", size = 16),
        axis.title = element_text(size = 17),
        axis.ticks = element_line(colour = "black", linewidth = .6),
        axis.ticks.length = unit(.22, "cm"))

stat_label <- function(xv, yv) {
  s <- fit_stats(yv, xv)
  r2 <- sprintf("%.2f", s[["R2"]])
  if (s[["p"]] < 1e-4) paste0("italic(R)^2 == '", r2, "' * ',' ~ italic(p) < 0.0001")
  else paste0("italic(R)^2 == '", r2, "' * ',' ~ italic(p) == '", sprintf("%.4f", s[["p"]]), "'")
}

pan_xy <- function(yv, ylab, xv = fig_df$ms2, xlab = "MS2 load (log10 CPM)") {
  d <- fig_df; d$x <- xv; d$y <- yv
  ggplot(d, aes(x, y)) +
    geom_smooth(data = d[fit_set, ], method = "lm", formula = y ~ x, se = TRUE, linetype = "dashed",
                colour = "grey20", fill = "grey85", linewidth = .5) +
    geom_point(aes(fill = group, alpha = fade), shape = 21, size = 3, colour = "black", stroke = .4) +
    scale_fill_manual(values = PAL) +
    scale_alpha_manual(values = c(normal = 0.95, faded = 0.4), guide = "none") +
    guides(fill = guide_legend(override.aes = list(alpha = 1))) +
    annotate("text", x = Inf, y = -Inf, label = stat_label(xv, yv), parse = TRUE,
             hjust = 1.08, vjust = -0.8, size = 5.2) +
    labs(x = xlab, y = ylab) + base_theme
}

combine2 <- function(pa, pb, name, stack = TRUE, shared_x = "MS2 load (log10 CPM)") {
  if (stack) {
    p <- (pa + theme(axis.title.x = element_blank(), legend.position = "none")) / pb
    save_fig(p, name, 6.5, 9.5)
  } else {
    no_x <- theme(axis.title.x = element_blank())
    xlab <- wrap_elements(panel = grid::textGrob(shared_x, gp = grid::gpar(fontsize = 17)))
    p <- (pa + no_x) + (pb + no_x) + xlab + guide_area() +
      plot_layout(design = "AB\nCC\nDD", guides = "collect", heights = c(1, 0.07, 0.08)) &
      theme(legend.direction = "horizontal")
    save_fig(p, name, 12, 6.2)
  }
}

## Figure 4C

In [ ]:
p4C <- pan_xy(fig_df$plasmid_log10cpm, "Plasmid expression\n(mean log10 CPM)")
save_fig(p4C, "Fig4C", 6.5, 5.4)
fit_stats(fig_df$plasmid_log10cpm)

## Figure 4D

In [ ]:
p4D <- pan_xy(fig_df$pga_log10cpm, "pga operon expression\n(mean log10 CPM)")
save_fig(p4D, "Fig4D", 6.5, 5.4)
fit_stats(fig_df$pga_log10cpm)

### Statistics reported in the text

In [ ]:
norm_counts <- counts(dds, normalized = TRUE)
bm <- rowMeans(norm_counts[other_genes, , drop = FALSE])
other_top <- names(sort(bm, decreasing = TRUE))[seq_len(min(length(pilus_genes), length(bm)))]
fig_df$other_top_log10cpm <- mean_log_cpm(other_top)

cat("correlation pilus vs other:", round(cor(fig_df$pilus_log10cpm[fit_set], fig_df$other_log10cpm[fit_set]), 3), "\n")
print(summary(lm(ms2 ~ pilus_log10cpm + other_log10cpm, data = fig_df[fit_set, ])))

pp <- fit_set & fig_df$plasmid_state == "plasmid_pos"
cat("\nplasmid-bearing isolates only, n =", sum(pp), "\n")
print(rbind(pilus = fit_stats(fig_df$pilus_log10cpm, keep = pp),
            other = fit_stats(fig_df$other_log10cpm, keep = pp),
            other_top = fit_stats(fig_df$other_top_log10cpm)))

## Figure S12A

In [ ]:
combine2(pan_xy(fig_df$pilus_log10cpm, "Pilus-associated genes\n(mean log10 CPM)"),
         pan_xy(fig_df$other_log10cpm, "Other plasmid genes\n(mean log10 CPM)"),
         "FigS12A", stack = FALSE)
rbind(pilus = fit_stats(fig_df$pilus_log10cpm), other = fit_stats(fig_df$other_log10cpm))

## Figure S12B

In [ ]:
XLAB_NEG <- "MS2 negative-strand load (log10 CPM)"
combine2(pan_xy(fig_df$pilus_log10cpm, "Pilus-associated genes\n(mean log10 CPM)", xv = fig_df$ms2_neg, xlab = XLAB_NEG),
         pan_xy(fig_df$other_log10cpm, "Other plasmid genes\n(mean log10 CPM)", xv = fig_df$ms2_neg, xlab = XLAB_NEG),
         "FigS12B", stack = FALSE, shared_x = XLAB_NEG)
cat("correlation, positive vs negative-strand MS2 load:", round(cor(fig_df$ms2[fit_set], fig_df$ms2_neg[fit_set]), 3), "\n")
print(summary(lm(ms2_neg ~ pilus_log10cpm + other_log10cpm, data = fig_df[fit_set, ])))

write.csv(fig_df, file.path(SRC_DIR, "Fig4CD_S12_data.csv"), row.names = FALSE)

## Figure S13

In [ ]:
rld <- rlog(dds, blind = TRUE)
top <- order(rowVars(assay(rld)), decreasing = TRUE)[1:500]
pca <- prcomp(t(assay(rld)[top, ]))
pct <- round(100 * summary(pca)$importance[2, ])

pc <- data.frame(sample = colnames(rld), pca$x[, 1:2])
pc$isolate <- ifelse(pc$sample %in% CONTROL_SAMPLES, "Control", sub("[abc]$", "", pc$sample))
pc$colour_group <- factor(ifelse(pc$isolate %in% c("Control", "i1", "i2", "i3", "i4"), pc$isolate, "Other"),
                          levels = c("Control", "i1", "i2", "i3", "i4", "Other"))
pc$condition <- ifelse(pc$sample %in% CONTROL_SAMPLES, "Control", "Isolate")

group_colors <- c(Control = "#E41A1C", i1 = "#984EA3", i2 = "#FF7F00", i3 = "#377EB8", i4 = "#4DAF4A", Other = "#A6A6A6")

pS13 <- ggplot(pc, aes(PC1, PC2, color = colour_group, shape = condition)) +
  geom_point(size = 4, alpha = 0.8) +
  geom_text_repel(aes(label = sample), size = 4, max.overlaps = Inf, show.legend = FALSE) +
  labs(x = paste0("PC1: ", pct[1], "% variance"), y = paste0("PC2: ", pct[2], "% variance")) +
  scale_color_manual(values = group_colors, name = "Group") +
  guides(shape = "none", color = guide_legend(override.aes = list(size = 6))) +
  theme_classic(base_size = 16) +
  theme(axis.title = element_text(size = 18),
        axis.text = element_text(color = "black", size = 14),
        axis.line = element_blank(),
        axis.ticks = element_line(color = "black", linewidth = 1),
        panel.border = element_rect(color = "black", fill = NA, linewidth = 1),
        legend.title = element_text(size = 20),
        legend.text = element_text(size = 17),
        legend.key.height = unit(0.65, "cm"),
        legend.key.width = unit(1.0, "cm"),
        legend.key.spacing.y = unit(0.1, "cm"))

save_fig(pS13, "FigS13", 9, 6)
write.csv(pc, file.path(SRC_DIR, "FigS13_data.csv"), row.names = FALSE)